# Most Probable R_L from the AB EEC Component

This notebook reads the pipeline `eec.parquet` output, isolates the AB EEC component, and extracts a most-probable `R_L` from a log-normal-like fit.

The stored column `eec_AB` is a density in physical opening angle, `dSigma_EEC / dR_L`, while the analysis bins are uniform in `ln(R_L)`. The extraction below fits `eec_AB` directly with a log-normal-like peak, modeled as a Gaussian in `ln(R_L)` plus an optional baseline. Its fitted peak is reported as `R_L_mode_eec`.

No extra `R_L` factor is applied in the fit or in the y-axis shown below.

In [ ]:
from pathlib import Path
import csv
import math
import os

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

try:
    from scipy.optimize import curve_fit
except ImportError:
    curve_fit = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.yaml").exists() and (PROJECT_ROOT.parent / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "config.yaml"
FIT_WINDOW_RL = None  # e.g. (0.02, 0.25); None uses all positive AB bins
MIN_POSITIVE_BINS = 5
BOOTSTRAP_REPLICAS = 100
BOOTSTRAP_SEED = 12345
BOOTSTRAP_REFIT = False  # True is slower; useful only after narrowing filters/window.
GRID_N_MU = 45
GRID_N_SIGMA = 30
FIT_BASELINE = True
FIT_STAGE1_POINTS_LEFT = 3
FIT_STAGE1_POINTS_RIGHT = 5
FIT_STAGE2_EXTRA_LEFT = 2
FIT_STAGE2_EXTRA_RIGHT = 5
FIT_STAGE2_ITERATIONS = 3
FIT_STAGE2_ITER_EXTRA_LEFT = 1
FIT_STAGE2_ITER_EXTRA_RIGHT = 2

# Optional filters. Leave as None to process everything in eec.parquet.
SAMPLES = None          # e.g. ["jewel_vac", "jewel_med"]
SELECTIONS = None       # e.g. ["softdrop", "maxkt"]
NORMALIZATIONS = None   # e.g. ["jet_pt2", "radiator_pt2"]

with CONFIG_PATH.open("r", encoding="utf-8") as stream:
    config = yaml.safe_load(stream)
LNDR_RANGE = (float(config["eec"]["lndr_min"]), float(config["eec"]["lndr_max"]))

output_dir = Path(config["output_dir"])
if not output_dir.is_absolute():
    output_dir = PROJECT_ROOT / output_dir

eec_path = output_dir / "eec.parquet"
summary_path = output_dir / "ab_most_probable_rl.csv"
summary_table_path = output_dir / "ab_fit_summary.csv"
eec_path

In [ ]:
def load_eec_rows(path):
    columns = [
        "sample",
        "selection",
        "normalization",
        "bin_center_lndr",
        "eec_AB",
        "eec_AB_err",
        "n_jets",
    ]
    return pq.read_table(path, columns=columns).to_pylist()


def keep_row(row):
    if SAMPLES is not None and row["sample"] not in SAMPLES:
        return False
    if SELECTIONS is not None and row["selection"] not in SELECTIONS:
        return False
    if NORMALIZATIONS is not None and row["normalization"] not in NORMALIZATIONS:
        return False
    return True


def grouped_rows(rows):
    groups = {}
    for row in rows:
        if not keep_row(row):
            continue
        key = (row["sample"], row["selection"], row["normalization"])
        groups.setdefault(key, []).append(row)
    return {key: sorted(value, key=lambda row: row["bin_center_lndr"]) for key, value in groups.items()}


rows = load_eec_rows(eec_path)
groups = grouped_rows(rows)
len(rows), len(groups)

## Fit Model

For each `(sample, selection, normalization)` group:

- Let `x = ln(R_L)`.
- Fit the stored AB EEC density directly: `eec_AB = baseline + amplitude * exp(-0.5 * ((x - mu) / sigma)^2)`.
- Initialize `mu` from the global maximum of the AB histogram in `ln(R_L)`.
- Initialize `sigma` from the weighted width around that starting peak.
- Run a two-stage fit: first fit only `FIT_STAGE1_POINTS_LEFT/RIGHT` bins around the MPV, then try up to `FIT_STAGE2_ITERATIONS` wider second-stage windows from that seed and keep the candidate with the best `chi2_ndf`.
- Use `scipy.optimize.curve_fit` when scipy is installed; otherwise use a deterministic numpy grid-search fallback.
- Bootstrap the binned AB spectrum using `eec_AB_err` and quote the spread of `R_L_mode_eec = exp(mu)`.
- By default `BOOTSTRAP_REFIT = False`, so bootstrap replicas use the fast global-maximum estimator. Set it to `True` only after narrowing the sample/selection/normalization or fit window if full refits are needed.

Use `FIT_WINDOW_RL` above to restrict the fit when the distribution has tails, shoulders, or low-statistics edge bins. The display x-axis is fixed to the configured EEC range, `config["eec"]["lndr_min"]` to `config["eec"]["lndr_max"]`, matching the production plots.

In [ ]:
def finite_positive_arrays(group_rows):
    x = np.asarray([row["bin_center_lndr"] for row in group_rows], dtype=float)
    rl = np.exp(x)
    y = np.asarray([row["eec_AB"] for row in group_rows], dtype=float)
    yerr = np.asarray([row["eec_AB_err"] for row in group_rows], dtype=float)

    mask = np.isfinite(x) & np.isfinite(y) & (y > 0)
    if FIT_WINDOW_RL is not None:
        lo, hi = FIT_WINDOW_RL
        mask &= (rl >= float(lo)) & (rl <= float(hi))
    return x[mask], rl[mask], y[mask], yerr[mask]


def eec_gaussian_lnr(x, amplitude, mu, sigma, baseline):
    sigma = np.maximum(sigma, 1.0e-9)
    return baseline + amplitude * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def global_peak_index(y):
    if len(y) == 0 or not np.any(np.isfinite(y)):
        return None
    y = np.asarray(y, dtype=float)
    y_work = np.where(np.isfinite(y), np.maximum(y, 0.0), 0.0)
    if not np.any(y_work > 0):
        return None
    return int(np.nanargmax(y_work))


def start_signal_and_peak(y):
    baseline0 = float(np.nanpercentile(y, 10.0)) if FIT_BASELINE else 0.0
    signal = np.clip(y - baseline0, 0.0, None)
    peak_idx = global_peak_index(signal)
    if np.sum(signal) <= 0:
        signal = np.clip(y, 0.0, None)
        peak_idx = global_peak_index(signal)
    return baseline0, signal, peak_idx


def indices_around_peak(n, peak_idx, left, right, min_points=MIN_POSITIVE_BINS):
    peak_idx = int(np.clip(peak_idx, 0, n - 1))
    lo = max(0, peak_idx - int(left))
    hi = min(n, peak_idx + int(right) + 1)
    while hi - lo < min(int(min_points), n):
        if lo > 0:
            lo -= 1
        if hi - lo >= min(int(min_points), n):
            break
        if hi < n:
            hi += 1
        if lo == 0 and hi == n:
            break
    return np.arange(lo, hi, dtype=int)


def histogram_start_parameters(x, y, peak_idx=None):
    baseline0, signal, found_peak_idx = start_signal_and_peak(y)
    if peak_idx is None:
        peak_idx = found_peak_idx
    if np.sum(signal) <= 0:
        mu0 = float(x[np.nanargmax(y)])
        sigma0 = max((float(np.max(x)) - float(np.min(x))) / 4.0, 1.0e-3)
    else:
        peak_idx = int(peak_idx) if peak_idx is not None else int(np.nanargmax(signal))
        mu0 = float(x[peak_idx])
        local_idx = indices_around_peak(len(x), peak_idx, FIT_STAGE1_POINTS_LEFT, FIT_STAGE1_POINTS_RIGHT)
        local_signal = signal[local_idx]
        if np.sum(local_signal) > 0:
            var0 = float(np.sum(local_signal * (x[local_idx] - mu0) ** 2) / np.sum(local_signal))
        else:
            var0 = float(np.sum(signal * (x - mu0) ** 2) / np.sum(signal))
        sigma0 = max(math.sqrt(max(var0, 0.0)), 1.0e-3)
    amplitude0 = max(float(np.nanmax(y) - baseline0), 1.0e-12)
    return np.asarray([amplitude0, mu0, sigma0, max(baseline0, 0.0)], dtype=float)


def y_uncertainties(y, yerr):
    sigma = np.asarray(yerr, dtype=float).copy()
    good = np.isfinite(sigma) & (sigma > 0)
    fallback = max(0.05 * float(np.nanmax(y)), 1.0e-12)
    sigma[~good] = fallback
    return np.maximum(sigma, 1.0e-12)


def fit_one_spectrum(x, y, yerr, p0=None):
    if p0 is None:
        p0 = histogram_start_parameters(x, y)
    sig_y = y_uncertainties(y, yerr)
    x_min = float(np.min(x))
    x_max = float(np.max(x))
    x_span = max(x_max - x_min, 1.0e-3)
    sigma_min = max(x_span / (20.0 * max(len(x), 1)), 1.0e-3)
    lower = [0.0, x_min, sigma_min, 0.0]
    upper = [np.inf, x_max, 2.0 * x_span, max(float(np.nanmax(y)), 1.0e-12)]

    if curve_fit is not None:
        popt, pcov = curve_fit(
            eec_gaussian_lnr,
            x,
            y,
            p0=np.clip(p0, lower, upper),
            sigma=sig_y,
            absolute_sigma=False,
            bounds=(lower, upper),
            maxfev=20000,
        )
        method = "scipy_curve_fit"
    else:
        popt, pcov = fit_one_spectrum_grid(x, y, sig_y, p0, lower, upper)
        method = "numpy_grid"
    return popt, pcov, method


def fit_one_spectrum_grid(x, y, sig_y, p0, lower, upper):
    x_min, x_max = lower[1], upper[1]
    x_span = max(x_max - x_min, 1.0e-3)
    mu0 = float(np.clip(p0[1], x_min, x_max))
    sigma0 = float(np.clip(p0[2], lower[2], upper[2]))
    mu_lo = max(x_min, mu0 - 0.75 * x_span)
    mu_hi = min(x_max, mu0 + 0.75 * x_span)
    sigma_lo = max(lower[2], sigma0 / 3.0)
    sigma_hi = min(upper[2], sigma0 * 3.0)
    if sigma_hi <= sigma_lo:
        sigma_lo, sigma_hi = lower[2], upper[2]
    mu_grid = np.linspace(mu_lo, mu_hi, int(GRID_N_MU))
    sigma_grid = np.geomspace(sigma_lo, sigma_hi, int(GRID_N_SIGMA))
    w = 1.0 / np.maximum(sig_y, 1.0e-12)
    best = None
    for mu in mu_grid:
        for sigma in sigma_grid:
            g = np.exp(-0.5 * ((x - mu) / sigma) ** 2)
            if FIT_BASELINE:
                design = np.column_stack([g, np.ones_like(g)])
            else:
                design = g[:, None]
            coeffs, *_ = np.linalg.lstsq(design * w[:, None], y * w, rcond=None)
            amplitude = max(float(coeffs[0]), 0.0)
            baseline = max(float(coeffs[1]), 0.0) if FIT_BASELINE and len(coeffs) > 1 else 0.0
            baseline = min(baseline, upper[3])
            pred = eec_gaussian_lnr(x, amplitude, mu, sigma, baseline)
            chi2 = float(np.sum(((y - pred) / np.maximum(sig_y, 1.0e-12)) ** 2))
            if best is None or chi2 < best[0]:
                best = (chi2, np.asarray([amplitude, mu, sigma, baseline], dtype=float))
    return best[1], None


def fit_chi2_ndf(x, y, yerr, popt):
    pred = eec_gaussian_lnr(x, *popt)
    sig_y = y_uncertainties(y, yerr)
    ndf = max(len(x) - len(popt), 1)
    return float(np.sum(((y - pred) / sig_y) ** 2) / ndf)


def bootstrap_modes(x, y, yerr, p0, rng):
    sig_y = y_uncertainties(y, yerr)
    modes = []
    for _ in range(int(BOOTSTRAP_REPLICAS)):
        y_boot = rng.normal(y, sig_y)
        y_boot = np.clip(y_boot, 0.0, None)
        if np.count_nonzero(y_boot > 0) < MIN_POSITIVE_BINS:
            continue
        try:
            if BOOTSTRAP_REFIT:
                popt, _, _ = fit_one_spectrum(x, y_boot, sig_y, p0=p0)
                mu = float(popt[1])
            else:
                mu = float(histogram_start_parameters(x, y_boot)[1])
        except Exception:
            continue
        modes.append(float(np.exp(mu)))
    return np.asarray(modes, dtype=float)


def fit_log_normal_like(group_rows, min_positive_bins=MIN_POSITIVE_BINS):
    x, rl, y, yerr = finite_positive_arrays(group_rows)
    if len(x) < min_positive_bins:
        return {"status": "too_few_positive_bins", "n_fit_bins": len(x)}
    _, signal, peak_idx = start_signal_and_peak(y)
    if peak_idx is None:
        return {"status": "no_start_peak", "n_fit_bins": len(x)}
    p0 = histogram_start_parameters(x, y, peak_idx=peak_idx)
    stage1_idx = indices_around_peak(len(x), peak_idx, FIT_STAGE1_POINTS_LEFT, FIT_STAGE1_POINTS_RIGHT)
    try:
        popt1, pcov1, method1 = fit_one_spectrum(x[stage1_idx], y[stage1_idx], yerr[stage1_idx], p0=p0)
    except Exception as exc:
        return {"status": f"stage1_fit_failed: {type(exc).__name__}", "n_fit_bins": len(x)}

    best = None
    p_iter = popt1
    method2 = None
    for iteration in range(int(FIT_STAGE2_ITERATIONS)):
        left = FIT_STAGE1_POINTS_LEFT + FIT_STAGE2_EXTRA_LEFT + iteration * FIT_STAGE2_ITER_EXTRA_LEFT
        right = FIT_STAGE1_POINTS_RIGHT + FIT_STAGE2_EXTRA_RIGHT + iteration * FIT_STAGE2_ITER_EXTRA_RIGHT
        candidate_idx = indices_around_peak(len(x), peak_idx, left, right)
        try:
            popt_candidate, pcov_candidate, method_candidate = fit_one_spectrum(
                x[candidate_idx], y[candidate_idx], yerr[candidate_idx], p0=p_iter
            )
        except Exception:
            continue
        x_candidate = x[candidate_idx]
        y_candidate = y[candidate_idx]
        yerr_candidate = yerr[candidate_idx]
        chi2_candidate = fit_chi2_ndf(x_candidate, y_candidate, yerr_candidate, popt_candidate)
        candidate = {
            "iteration": int(iteration + 1),
            "idx": candidate_idx,
            "popt": popt_candidate,
            "pcov": pcov_candidate,
            "method": method_candidate,
            "chi2_ndf": chi2_candidate,
            "left": int(left),
            "right": int(right),
        }
        if best is None or chi2_candidate < best["chi2_ndf"]:
            best = candidate
        p_iter = popt_candidate

    if best is None:
        return {"status": "stage2_fit_failed", "n_fit_bins": len(x)}

    stage2_idx = best["idx"]
    popt = best["popt"]
    method2 = best["method"]
    chi2_ndf = best["chi2_ndf"]
    x_fit = x[stage2_idx]
    y_fit = y[stage2_idx]
    yerr_fit = yerr[stage2_idx]
    rng = np.random.default_rng(int(BOOTSTRAP_SEED))
    boot = bootstrap_modes(x_fit, y_fit, yerr_fit, popt, rng)

    rl_mode_eec = float(np.exp(popt[1]))
    rl_mode_eec_err = float(np.std(boot, ddof=1)) if len(boot) > 1 else math.nan
    return {
        "status": "ok",
        "fit_method": f"{method2}_two_step",
        "fit_stage1_method": method1,
        "n_fit_bins": int(len(stage2_idx)),
        "n_stage1_bins": int(len(stage1_idx)),
        "fit_iteration": int(best["iteration"]),
        "fit_iterations_tried": int(FIT_STAGE2_ITERATIONS),
        "fit_points_left": int(best["left"]),
        "fit_points_right": int(best["right"]),
        "peak_index": int(peak_idx),
        "fit_stage1_x_min": float(np.min(x[stage1_idx])),
        "fit_stage1_x_max": float(np.max(x[stage1_idx])),
        "fit_x_min": float(np.min(x_fit)),
        "fit_x_max": float(np.max(x_fit)),
        "fit_rl_min": float(np.exp(np.min(x_fit))),
        "fit_rl_max": float(np.exp(np.max(x_fit))),
        "amplitude": float(popt[0]),
        "mu_lnr": float(popt[1]),
        "sigma_lnr": float(abs(popt[2])),
        "baseline": float(popt[3]),
        "stage1_amplitude": float(popt1[0]),
        "stage1_mu_lnr": float(popt1[1]),
        "stage1_sigma_lnr": float(abs(popt1[2])),
        "stage1_baseline": float(popt1[3]),
        "mu_lnr_start": float(p0[1]),
        "sigma_lnr_start": float(p0[2]),
        "rl_mode_eec": rl_mode_eec,
        "rl_mode_eec_err": rl_mode_eec_err,
        "rl_mode_eec_boot_p16": float(np.percentile(boot, 16.0)) if len(boot) else math.nan,
        "rl_mode_eec_boot_p84": float(np.percentile(boot, 84.0)) if len(boot) else math.nan,
        "bootstrap_replicas_ok": int(len(boot)),
        "bootstrap_mode": "refit" if BOOTSTRAP_REFIT else "global_max",
        "chi2_ndf": chi2_ndf,
    }


results = []
group_items = sorted(groups.items())
for i, ((sample, selection, normalization), group) in enumerate(group_items, start=1):
    print(f"[{i}/{len(group_items)}] fitting {sample}, {selection}, {normalization}")
    result = fit_log_normal_like(group)
    result.update({"sample": sample, "selection": selection, "normalization": normalization})
    results.append(result)

len(results)

In [ ]:
def print_results_table(results):
    columns = [
        "sample",
        "selection",
        "normalization",
        "status",
        "fit_method",
        "n_stage1_bins",
        "n_fit_bins",
        "fit_iteration",
        "fit_points_left",
        "fit_points_right",
        "rl_mode_eec",
        "rl_mode_eec_err",
        "rl_mode_eec_boot_p16",
        "rl_mode_eec_boot_p84",
        "bootstrap_mode",
        "mu_lnr_start",
        "stage1_mu_lnr",
        "chi2_ndf",
        "sigma_lnr",
    ]
    rows = []
    for result in results:
        row = []
        for column in columns:
            value = result.get(column, "")
            if isinstance(value, float):
                row.append(f"{value:.6g}" if np.isfinite(value) else "")
            else:
                row.append(str(value))
        rows.append(row)
    widths = [max(len(column), *(len(row[i]) for row in rows)) for i, column in enumerate(columns)]
    print("  ".join(column.ljust(widths[i]) for i, column in enumerate(columns)))
    print("  ".join("-" * width for width in widths))
    for row in rows:
        print("  ".join(row[i].ljust(widths[i]) for i in range(len(columns))))


print_results_table(results)

## Fit Summary Table

`summary_df` joins the fit results to selected-splitting kinematics. Fit mean and width are kept as separate columns: `mu_lnr` and `sigma_lnr`. Radiator pT uses `parent_pt`, i.e. the fjcontrib selected radiator `pair().perp()`, with fallback to `pt_a + pt_b` if unavailable.

In [ ]:
def load_radiator_pt_summary(config):
    rows = []
    columns = ["selection", "jet_pt", "pt_a", "pt_b", "parent_pt"]
    for sample in config["samples"]:
        sample_name = sample["name"]
        path = output_dir / f"{sample_name}__splittings.parquet"
        if not path.exists():
            continue
        table = pq.read_table(path, columns=columns).to_pandas()
        parent = table["parent_pt"].to_numpy(dtype=float)
        fallback = (table["pt_a"] + table["pt_b"]).to_numpy(dtype=float)
        radiator_pt = np.where(np.isfinite(parent) & (parent > 0), parent, fallback)
        table = table.assign(sample=sample_name, radiator_pt=radiator_pt)
        for selection, group in table.groupby("selection", sort=True):
            values = group["radiator_pt"].to_numpy(dtype=float)
            values = values[np.isfinite(values) & (values > 0)]
            jet_values = group["jet_pt"].to_numpy(dtype=float)
            jet_values = jet_values[np.isfinite(jet_values) & (jet_values > 0)]
            n = int(len(values))
            rows.append(
                {
                    "sample": sample_name,
                    "selection": selection,
                    "n_splittings": n,
                    "radiator_pt_mean": float(np.mean(values)) if n else np.nan,
                    "radiator_pt_median": float(np.median(values)) if n else np.nan,
                    "radiator_pt_std": float(np.std(values, ddof=1)) if n > 1 else np.nan,
                    "radiator_pt_sem": float(np.std(values, ddof=1) / np.sqrt(n)) if n > 1 else np.nan,
                    "jet_pt_mean": float(np.mean(jet_values)) if len(jet_values) else np.nan,
                }
            )
    return pd.DataFrame(rows)


fit_df = pd.DataFrame(results)
radiator_pt_df = load_radiator_pt_summary(config)
summary_df = fit_df.merge(radiator_pt_df, on=["sample", "selection"], how="left")
summary_df["rl_mode_eec_minus"] = summary_df["rl_mode_eec"] - summary_df["rl_mode_eec_boot_p16"]
summary_df["rl_mode_eec_plus"] = summary_df["rl_mode_eec_boot_p84"] - summary_df["rl_mode_eec"]
summary_rows = summary_df.to_dict("records")
summary_by_key = {
    (row["sample"], row["selection"], row["normalization"]): row
    for row in summary_rows
}
summary_df.to_csv(summary_table_path, index=False)

display_columns = [
    "sample",
    "selection",
    "normalization",
    "radiator_pt_mean",
    "mu_lnr",
    "sigma_lnr",
    "rl_mode_eec",
    "rl_mode_eec_err",
    "chi2_ndf",
    "fit_iteration",
]
summary_df[display_columns].sort_values(["sample", "normalization", "selection"])

## Configurable Summary Plot

Edit the variables in this cell to choose what is plotted. The default uses radiator pT on the x-axis and shows the fit mean (`mu_lnr`) and width (`sigma_lnr`) separately, grouped by splitting selection.

In [ ]:
PLOT_X = "radiator_pt_mean"
PLOT_Y_FIELDS = ["mu_lnr", "sigma_lnr"]
PLOT_GROUP_FIELD = "selection"
PLOT_LABEL_FIELD = "sample"
PLOT_FILTERS = {
    "normalization": "jet_pt2",
    # "sample": "jewel_vac",
    # "selection": ["maxkt", "sd_z0p1", "sd_z0p2", "sd_z0p4"],
}
PLOT_YERR = {
    "rl_mode_eec": "rl_mode_eec_err",
    "mu_lnr": None,
    "sigma_lnr": None,
}


def filtered_summary(df, filters):
    out = df.copy()
    for key, value in filters.items():
        if value is None:
            continue
        if isinstance(value, (list, tuple, set)):
            out = out[out[key].isin(value)]
        else:
            out = out[out[key] == value]
    return out


plot_df = filtered_summary(summary_df, PLOT_FILTERS)
fig, axes = plt.subplots(1, len(PLOT_Y_FIELDS), figsize=(6 * len(PLOT_Y_FIELDS), 4), squeeze=False)
for ax, y_field in zip(axes.ravel(), PLOT_Y_FIELDS):
    for group_value, group in plot_df.groupby(PLOT_GROUP_FIELD, sort=True):
        group = group.sort_values(PLOT_X)
        yerr_field = PLOT_YERR.get(y_field)
        yerr = group[yerr_field].to_numpy(dtype=float) if yerr_field else None
        ax.errorbar(
            group[PLOT_X].to_numpy(dtype=float),
            group[y_field].to_numpy(dtype=float),
            yerr=yerr,
            marker="o",
            lw=1,
            label=str(group_value),
        )
        for _, row in group.iterrows():
            if PLOT_LABEL_FIELD:
                ax.annotate(str(row[PLOT_LABEL_FIELD]), (row[PLOT_X], row[y_field]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel(PLOT_X)
    ax.set_ylabel(y_field)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, title=PLOT_GROUP_FIELD)
fig.tight_layout()
fig

In [ ]:
fieldnames = [
    "sample",
    "selection",
    "normalization",
    "status",
    "peak_index",
    "n_stage1_bins",
    "n_fit_bins",
    "fit_iteration",
    "fit_iterations_tried",
    "fit_points_left",
    "fit_points_right",
    "fit_stage1_x_min",
    "fit_stage1_x_max",
    "fit_x_min",
    "fit_x_max",
    "fit_rl_min",
    "fit_rl_max",
    "mu_lnr",
    "sigma_lnr",
    "baseline",
    "amplitude",
    "stage1_mu_lnr",
    "stage1_sigma_lnr",
    "stage1_baseline",
    "stage1_amplitude",
    "mu_lnr_start",
    "sigma_lnr_start",
    "rl_mode_eec",
    "rl_mode_eec_err",
    "rl_mode_eec_boot_p16",
    "rl_mode_eec_boot_p84",
    "bootstrap_replicas_ok",
    "bootstrap_mode",
    "chi2_ndf",
    "fit_method",
    "fit_stage1_method",
]

summary_path.parent.mkdir(parents=True, exist_ok=True)
with summary_path.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    for result in results:
        writer.writerow(result)

summary_path

In [ ]:
def plot_fit_for_group(key, group_rows, result, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))
    sample, selection, normalization = key
    x = np.asarray([row["bin_center_lndr"] for row in group_rows], dtype=float)
    rl = np.exp(x)
    y_dr = np.asarray([row["eec_AB"] for row in group_rows], dtype=float)
    yerr_dr = np.asarray([row["eec_AB_err"] for row in group_rows], dtype=float)

    valid = np.isfinite(x) & np.isfinite(y_dr) & (y_dr > 0)
    ax.errorbar(x[valid], y_dr[valid], yerr=yerr_dr[valid], fmt="o", ms=3, lw=1, label="AB EEC = dSigma/dR_L")

    if result.get("status") == "ok":
        ax.axvspan(result["fit_stage1_x_min"], result["fit_stage1_x_max"], color="0.85", alpha=0.35, label="stage-1 seed window")
        xx = np.linspace(result["fit_x_min"], result["fit_x_max"], 300)
        yy = eec_gaussian_lnr(xx, result["amplitude"], result["mu_lnr"], result["sigma_lnr"], result["baseline"])
        yy1 = eec_gaussian_lnr(xx, result["stage1_amplitude"], result["stage1_mu_lnr"], result["stage1_sigma_lnr"], result["stage1_baseline"])
        ax.plot(xx, yy1, color="0.45", ls="--", lw=1, label="stage-1 fit propagated")
        ax.plot(xx, yy, label=f"stage-2 Gaussian ({result['fit_method']})")
        ax.axvline(result["mu_lnr_start"], color="0.55", ls=":", lw=1, label="global maximum start")
        ax.axvline(result["mu_lnr"], color="black", ls="--", lw=1, label="mode of eec_AB")

    ax.set_yscale("linear")
    ax.set_xlim(*LNDR_RANGE)
    ax.set_xlabel("ln(R_L)")
    ax.set_ylabel("eec_AB = dSigma_EEC^AB / dR_L")
    ax.set_title(f"{sample}, {selection}, {normalization} (fit: eec_AB, no R_L factor)")
    ax.legend(fontsize=8)
    return ax


ok_results = [result for result in results if result.get("status") == "ok"]
ncols = 2
nrows = max(1, math.ceil(len(ok_results) / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(7 * ncols, 3.8 * nrows), squeeze=False)

for ax in axes.ravel():
    ax.axis("off")

for ax, result in zip(axes.ravel(), ok_results):
    key = (result["sample"], result["selection"], result["normalization"])
    ax.axis("on")
    plot_fit_for_group(key, groups[key], result, ax=ax)

fig.tight_layout()
fig

## Practical Checks

- If the fit status is not `ok`, inspect the AB spectrum and adjust `FIT_WINDOW_RL`.
- If the AB spectrum is monotonic in the available range, a log-normal mode is not constrained by these bins.
- Compare `R_L_mode_eec` across normalizations only when the same `(sample, selection)` and fit window are used.
- The output CSV is written next to the parquet outputs as `ab_most_probable_rl.csv`.